# Training frontier Model to beat Pricer With Lower Dataset Size

This notebook implements the step-by-step plan to train a price-prediction model that **beats** the week6 example (fine-tuned GPT-4.1-nano, ~$67.75 error on 20k examples) using the **same** Amazon Appliances pipeline but with a **smaller** training set (2k–5k samples), while respecting system capabilities via `system_info`.

In [ ]:
# Step 1: Environment and data subset
import os
import sys
import random
import json

# Add week6 to path so pricer can be imported when notebook runs from community-contributions/Chrys/
_week6_dir = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if _week6_dir not in sys.path:
    sys.path.insert(0, _week6_dir)

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import login
from sklearn.linear_model import LinearRegression
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from pricer.evaluator import evaluate
from pricer.items import Item
from tqdm.notebook import tqdm
try:
    from IPython.display import display
except ImportError:
    display = print

load_dotenv(override=True)
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(hf_token, add_to_git_credential=True)

In [ ]:
# System info: add week4 to path and get CPU/GPU capabilities
_repo_root = os.path.abspath(os.path.join(_week6_dir, ".."))
_week4 = os.path.join(_repo_root, "week4")
if os.path.isdir(_week4) and _week4 not in sys.path:
    sys.path.insert(0, _week4)
try:
    from system_info import retrieve_system_info
    SYSTEM_INFO = retrieve_system_info()
    print("System info:", json.dumps(SYSTEM_INFO, indent=2))
except Exception as e:
    SYSTEM_INFO = {}
    print("system_info not available:", e)

In [ ]:
# GPU/MPS and n_jobs from system_info
import torch
cores = SYSTEM_INFO.get("cpu", {}).get("cores_logical") or os.cpu_count() or 4
N_JOBS = max(1, min(4, (cores or 4) - 1))
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
BATCH_SIZE = 32  # reduce to 16 if OOM on 8GB RAM
print(f"N_JOBS={N_JOBS}, DEVICE={DEVICE}, BATCH_SIZE={BATCH_SIZE}")

In [ ]:
# Load data (same dataset as example) and define smaller training subset
username = "ed-donner"
dataset_name = f"{username}/items_lite"
train, val, test = Item.from_hub(dataset_name)
print(f"Loaded {len(train):,} train, {len(val):,} val, {len(test):,} test")

# Lower dataset size: use 3000 training samples (plan: 2k–5k)
TRAIN_SUBSET_SIZE = 20000
train_small = train[:TRAIN_SUBSET_SIZE]
print(f"Using train_small with {len(train_small):,} items for training. Val and test unchanged.")

## Establish example baseline

Run the pre-trained GPT-4.1-nano (no fine-tuning) to get a baseline error on the same test set. Requires OpenAI API. Skip this cell if you want to avoid API cost; you can still compare against the reported ~$67.75 (fine-tuned 20k) or the zero-shot nano error from the course.

In [ ]:
# Uncomment and run to get baseline (requires OPENAI_API_KEY / litellm). Record the "Error: $X.XX" for comparison.
# from litellm import completion
# def messages_for(item):
#     msg = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
#     return [{"role": "user", "content": msg}]
# def gpt_4_1_nano_baseline(item):
#     r = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
#     return r.choices[0].message.content
# evaluate(gpt_4_1_nano_baseline, test, size=50)  # use size=50 to save cost; or 200 for full
EXAMPLE_BASELINE_ERROR = 67.75  # from plan: fine-tuned nano 20k. Replace with your run if you evaluated.
print(f"Target: beat example baseline error ${EXAMPLE_BASELINE_ERROR:.2f}")

## Data quality and stratification

Create a stratified `train_small` so price distribution is balanced (cheap and expensive items both represented). Filter out items with missing or very short summaries.

In [ ]:
# Filter: keep only items with usable summary
def has_good_summary(item):
    s = (item.summary or "").strip()
    return len(s) >= 20
train_filtered = [x for x in train if has_good_summary(x)]
print(f"After filtering: {len(train_filtered):,} items (dropped {len(train) - len(train_filtered):,})")

### Optional: Local Ollama model (e.g. qwen3-vl:4b)

Use a model running locally via [Ollama](https://ollama.com). Start the model first (e.g. `ollama run qwen3-vl:4b`), then run the cell below. Same prompt as the API baseline; no API cost.

In [ ]:
# Local Ollama model: ensure the model is running (e.g. ollama run qwen3-vl:4b)
from litellm import completion

OLLAMA_MODEL = "llama3.2"  # or e.g. "llama3.2", "qwen2.5:7b"
OLLAMA_BASE = "http://localhost:11434"

def messages_for(item):
    msg = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": msg}]

def local_ollama_pricer(item):
    r = completion(
        model=f"ollama/{OLLAMA_MODEL}",
        messages=messages_for(item),
        api_base=OLLAMA_BASE,
    )
    return r.choices[0].message.content
# Uncomment to evaluate (use small size first; local inference can be slow)
# random.seed(42)
# evaluate(local_ollama_pricer, test, size=50)


In [ ]:
# Stratified sampling by price bins
def stratified_sample(items, n, bins=(0, 50, 150, 500, 1000), random_state=42):
    rng = np.random.default_rng(random_state)
    by_bin = {}
    for item in items:
        p = item.price
        b = next((i for i in range(len(bins) - 1) if bins[i] <= p < bins[i + 1]), len(bins) - 2)
        by_bin.setdefault(b, []).append(item)
    out = []
    for b in sorted(by_bin.keys()):
        pool = by_bin[b]
        k = max(1, round(n * len(pool) / len(items)))
        k = min(k, len(pool))
        idx = rng.choice(len(pool), size=k, replace=False)
        out.extend([pool[i] for i in idx])
    rng.shuffle(out)
    return out[:n]

train_small = stratified_sample(train_filtered, min(TRAIN_SUBSET_SIZE, len(train_filtered)))
print(f"train_small (stratified): {len(train_small):,} items")

## Traditional ML (TF-IDF + XGBoost, Linear Regression)

Use text (TF-IDF) plus optional numeric features. Train XGBoost and Linear Regression on `train_small`; evaluate on the same test set.

In [ ]:
def get_features(item):
    return {
        "weight": item.weight if item.weight is not None else 0.0,
        "weight_unknown": 1 if (item.weight is None or item.weight == 0) else 0,
        "text_length": len(item.summary or ""),
    }

def list_to_dataframe(items):
    features = [get_features(item) for item in items]
    df = pd.DataFrame(features)
    df["price"] = [item.price for item in items]
    return df

In [ ]:
# TF-IDF on summaries (train_small only)
np.random.seed(42)
documents_train = [item.summary or "" for item in train_small]
tfidf = TfidfVectorizer(max_features=3000, stop_words="english")
X_tfidf_train = tfidf.fit_transform(documents_train)
X_tfidf_val = tfidf.transform([item.summary or "" for item in val])
X_tfidf_test = tfidf.transform([item.summary or "" for item in test])
print(f"TF-IDF shape: {X_tfidf_train.shape}")

In [ ]:
# Combine TF-IDF with numeric features
from scipy.sparse import hstack
train_df = list_to_dataframe(train_small)
val_df = list_to_dataframe(val)
test_df = list_to_dataframe(test)
num_cols = ["weight", "weight_unknown", "text_length"]
X_train_combined = hstack([X_tfidf_train, train_df[num_cols].values])
X_val_combined = hstack([X_tfidf_val, val_df[num_cols].values])
X_test_combined = hstack([X_tfidf_test, test_df[num_cols].values])
y_train = train_df["price"].values
y_val = val_df["price"].values
y_test = test_df["price"].values

In [ ]:
# Linear Regression baseline
lr_model = LinearRegression()
lr_model.fit(X_train_combined, y_train)
y_pred_lr = np.maximum(0, lr_model.predict(X_test_combined))
print(f"Linear Regression MSE: {mean_squared_error(y_test, y_pred_lr):.2f}, R²: {r2_score(y_test, y_pred_lr):.4f}")

In [ ]:
# XGBoost (fall back to RandomForest if unavailable or kernel crashes)
# If the kernel crashes here, set USE_XGBOOST = False below (or on Mac: brew install libomp)
USE_XGBOOST = False  # Set True to try XGBoost; False avoids loading it and prevents crash when libomp is missing
use_xgb = False
if USE_XGBOOST:
    try:
        import xgboost as xgb
        xgb_model = xgb.XGBRegressor(n_estimators=400, max_depth=7, learning_rate=0.07, n_jobs=N_JOBS, random_state=42)
        xgb_model.fit(X_train_combined, y_train, eval_set=[(X_val_combined, y_val)], verbose=False)
        use_xgb = True
    except Exception:
        pass
if not use_xgb:
    from sklearn.ensemble import RandomForestRegressor
    xgb_model = RandomForestRegressor(n_estimators=300, max_depth=12, n_jobs=N_JOBS, random_state=42)
    xgb_model.fit(X_train_combined, y_train)
print("XGBoost" if use_xgb else "RandomForest", "fitted.")

In [ ]:
# Pricer functions for evaluator (item -> predicted price)
def linear_regression_pricer(item):
    txt = tfidf.transform([item.summary or ""])
    num = np.array([get_features(item)[c] for c in num_cols]).reshape(1, -1)
    X = hstack([txt, num])
    return max(0.0, float(lr_model.predict(X)[0]))

def xgboost_pricer(item):
    txt = tfidf.transform([item.summary or ""])
    num = np.array([get_features(item)[c] for c in num_cols]).reshape(1, -1)
    X = hstack([txt, num])
    return max(0.0, float(xgb_model.predict(X)[0]))

In [ ]:
# Evaluate Traditional ML models
random.seed(42)
print("Linear Regression:")
evaluate(linear_regression_pricer, test)
print("\nXGBoost/RandomForest:")
evaluate(xgboost_pricer, test)

## Neural model (MLP with dropout)

Train a 4–6 layer MLP on the same text (vectorized) + numeric features. Uses dropout to reduce overfitting on small data. Device (CPU/MPS/CUDA) was set from system_info in Step 1.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

class MLPPricer(torch.nn.Module):
    def __init__(self, input_size, hidden=(256, 128, 64), dropout=0.25):
        super().__init__()
        layers = []
        prev = input_size
        for h in hidden:
            layers.extend([torch.nn.Linear(prev, h), torch.nn.ReLU(), torch.nn.Dropout(dropout)])
            prev = h
        layers.append(torch.nn.Linear(prev, 1))
        self.net = torch.nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [ ]:
# Prepare tensors (dense for PyTorch)
X_train_dense = torch.FloatTensor(X_train_combined.toarray())
y_train_t = torch.FloatTensor(y_train).unsqueeze(1)
X_val_dense = torch.FloatTensor(X_val_combined.toarray())
y_val_t = torch.FloatTensor(y_val).unsqueeze(1)
train_ds = TensorDataset(X_train_dense, y_train_t)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
input_size = X_train_dense.shape[1]
nn_model = MLPPricer(input_size).to(DEVICE)
optimizer = torch.optim.Adam(nn_model.parameters(), lr=1e-3)
loss_fn = torch.nn.MSELoss()
print(f"MLP parameters: {sum(p.numel() for p in nn_model.parameters()):,}")

In [ ]:
# Train neural network (5–10 epochs, early stop on val loss)
EPOCHS = 5
best_val = float("inf")
for epoch in range(EPOCHS):
    nn_model.train()
    for bx, by in train_loader:
        bx, by = bx.to(DEVICE), by.to(DEVICE)
        optimizer.zero_grad()
        loss = loss_fn(nn_model(bx), by)
        loss.backward()
        optimizer.step()
    nn_model.eval()
    with torch.no_grad():
        val_loss = loss_fn(nn_model(X_val_dense.to(DEVICE)), y_val_t.to(DEVICE)).item()
    if val_loss < best_val:
        best_val = val_loss
    print(f"Epoch {epoch+1}/{EPOCHS} Val MSE: {val_loss:.2f}")

In [ ]:
def neural_pricer(item):
    nn_model.eval()
    with torch.no_grad():
        txt = tfidf.transform([item.summary or ""])
        num = np.array([get_features(item)[c] for c in num_cols]).reshape(1, -1)
        X = hstack([txt, num]).toarray()
        x = torch.FloatTensor(X).to(DEVICE)
        out = nn_model(x)[0].item()
    return max(0.0, out)

random.seed(42)
print("Neural network (MLP):")
evaluate(neural_pricer, test)

## Evaluation and summary table

Summary of average absolute error (and optionally MSE, r²) for each model. Primary metric: **Error ($)** — lower is better. Goal: beat the example baseline (~$67.75).

In [ ]:
# Compute metrics for each pricer on test set (same size as evaluator default: 200)
def compute_metrics(pricer_fn, data, size=200):
    from pricer.evaluator import Tester
    t = Tester(pricer_fn, data, size=size, workers=5)
    for i in range(size):
        title, guess, truth, error, color = t.run_datapoint(i)
        t.titles.append(title)
        t.guesses.append(guess)
        t.truths.append(truth)
        t.errors.append(error)
        t.colors.append(color)
    avg_err = sum(t.errors) / len(t.errors)
    mse = mean_squared_error(t.truths, t.guesses)
    r2 = r2_score(t.truths, t.guesses) * 100
    return {"error": avg_err, "mse": mse, "r2": r2}

EVAL_SIZE = min(200, len(test))
results = []
for name, fn in [
    ("Linear Regression", linear_regression_pricer),
    ("XGBoost/RF", xgboost_pricer),
    ("Neural MLP", neural_pricer),
]:
    m = compute_metrics(fn, test, size=EVAL_SIZE)
    results.append({"model": name, "train_n": len(train_small), **m})
    print(f"{name}: Error ${m['error']:.2f}, MSE={m['mse']:.0f}, r²={m['r2']:.1f}%")

In [ ]:
# Summary table including example baseline
results.append({
    "model": "Example (fine-tuned nano 20k)",
    "train_n": 20_000,
    "error": EXAMPLE_BASELINE_ERROR,
    "mse": None,
    "r2": None,
})
summary_df = pd.DataFrame(results)
summary_df["beat_baseline"] = summary_df["error"].apply(lambda e: "Yes" if e < EXAMPLE_BASELINE_ERROR else "No")
display(summary_df[["model", "train_n", "error", "mse", "r2", "beat_baseline"]])
best = summary_df.loc[summary_df["error"].idxmin()]
print(f"\nBest model: {best['model']} (Error ${best['error']:.2f})")